# Visualize HyP3 InSAR Products

## Overview

Displays HyP3 interferogram products: unwrapped phase, wrapped phase, and correlation. Creates publication-ready multi-panel figures showing data quality and spatial patterns.

**Prerequisites:**
- HyP3 InSAR products downloaded and extracted
- `snowsar` package installed
- `rasterio` and `matplotlib` available

**HyP3 products visualized:**
- `*unw_phase.tif`: Unwrapped phase (continuous displacement)
- `*wrapped_phase.tif`: Wrapped phase (interferogram fringes, -π to π)
- `*corr.tif`: Correlation (coherence, 0-1 quality metric)

**Interpreting products:**
- **Unwrapped phase**: Direct measure of LOS displacement (radians → cm conversion depends on wavelength)
- **Wrapped phase**: Fringes indicate displacement gradient; one cycle = λ/2 displacement
- **Correlation**: Quality metric (> 0.7 = good, 0.4-0.7 = moderate, < 0.4 = noisy/decorrelated)

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from snowsar.utils import (
    parse_date_pairs_from_hyp3_filenames,
    read_geotiff_as_array,
    find_matching_products,
    plot_hyp3_trio,
    plot_correlation_histogram,
)

## Configuration

In [ ]:
# ============================================================================
# CONFIGURATION - Update these paths for your setup
# ============================================================================

# Directory containing HyP3 product folders
HYP3_DIR = Path("/Volumes/Fortress_L3/SnowWaterEquivalent/SWE_Shadi/06082024_hyp3/2020/Path_071_Frame_450")

# Glob patterns for each product type
# Adjust if your files have different naming conventions
UNWRAPPED_PATTERN = "*unw_phase.tif"
WRAPPED_PATTERN = "*wrapped_phase.tif"
CORRELATION_PATTERN = "*corr.tif"

# ============================================================================
# DISPLAY OPTIONS
# ============================================================================

# Color limits for each product (None = auto-scale)
# Adjust these based on your expected phase range
UNWRAPPED_VLIM_RAD = (-7, 7)  # Radians
CORRELATION_VLIM = (0, 1)     # Always 0-1

# Figure size for 3-panel display (width, height in inches)
FIGSIZE = (18, 9)

# DPI for saving figures (higher = better quality, larger file)
SAVE_DPI = 150

# ============================================================================
# Validation
# ============================================================================
if not HYP3_DIR.exists():
    raise FileNotFoundError(
        f"HyP3 directory not found: {HYP3_DIR}\n"
        f"Update HYP3_DIR to point to your HyP3 products directory"
    )

unwrapped_files = sorted(HYP3_DIR.rglob(UNWRAPPED_PATTERN))
wrapped_files = sorted(HYP3_DIR.rglob(WRAPPED_PATTERN))
correlation_files = sorted(HYP3_DIR.rglob(CORRELATION_PATTERN))

print(f"Found {len(unwrapped_files)} unwrapped phase files")
print(f"Found {len(wrapped_files)} wrapped phase files")
print(f"Found {len(correlation_files)} correlation files")

if len(unwrapped_files) == 0:
    raise FileNotFoundError(
        f"No unwrapped phase files found matching: {HYP3_DIR / '**' / UNWRAPPED_PATTERN}\n"
        f"Check that HYP3_DIR and UNWRAPPED_PATTERN are correct"
    )

## Single Interferogram Visualization

Display first interferogram as example. Modify the index to view different pairs.

In [ ]:
# Select interferogram to visualize (0 = first, 1 = second, etc.)
IFG_INDEX = 0

if len(unwrapped_files) > IFG_INDEX:
    unw_file = unwrapped_files[IFG_INDEX]
    wrapped_file, corr_file = find_matching_products(
        unw_file, wrapped_files, correlation_files
    )
    
    print(f"Visualizing interferogram {IFG_INDEX + 1}/{len(unwrapped_files)}:")
    print(f"  Unwrapped: {unw_file.name}")
    print(f"  Wrapped:   {wrapped_file.name if wrapped_file else 'Not found'}")
    print(f"  Correlation: {corr_file.name if corr_file else 'Not found'}")
    
    fig, axes = plot_hyp3_trio(
        unw_file,
        wrapped_file,
        corr_file,
        unw_vlim_rad=UNWRAPPED_VLIM_RAD,
        corr_vlim=CORRELATION_VLIM,
        figsize=FIGSIZE,
    )
    plt.show()
else:
    print(f"No interferogram at index {IFG_INDEX}. Found {len(unwrapped_files)} total.")

## Batch Visualization

Generate figures for all interferograms and save to output directory.

In [ ]:
# Output directory for saved figures
OUTPUT_DIR = Path("./outputs/hyp3_figures")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Processing {len(unwrapped_files)} interferograms...")

for idx, unw_file in enumerate(unwrapped_files):
    wrapped_file, corr_file = find_matching_products(
        unw_file, wrapped_files, correlation_files
    )
    
    # Parse dates for filename
    date_pairs = parse_date_pairs_from_hyp3_filenames([unw_file])
    if date_pairs:
        ref_date, sec_date = date_pairs[0]
        output_name = f"ifg_{ref_date.strftime('%Y%m%d')}_{sec_date.strftime('%Y%m%d')}.png"
    else:
        output_name = f"ifg_{idx:03d}.png"
    
    output_path = OUTPUT_DIR / output_name
    
    fig, axes = plot_hyp3_trio(
        unw_file,
        wrapped_file,
        corr_file,
        unw_vlim_rad=UNWRAPPED_VLIM_RAD,
        corr_vlim=CORRELATION_VLIM,
        figsize=FIGSIZE,
        save_path=output_path,
        dpi=SAVE_DPI,
    )
    plt.close(fig)  # Close to free memory
    
    if (idx + 1) % 5 == 0:
        print(f"  Processed {idx + 1}/{len(unwrapped_files)}")

print(f"\nAll figures saved to {OUTPUT_DIR}")

## Data Quality Assessment

Correlation histogram across all interferograms shows overall coherence and helps identify low-quality pairs.

In [ ]:
if len(correlation_files) > 0:
    fig, ax = plot_correlation_histogram(correlation_files)
    
    # Save histogram
    hist_path = OUTPUT_DIR / "correlation_histogram.png"
    fig.savefig(hist_path, dpi=SAVE_DPI, bbox_inches="tight")
    print(f"Saved correlation histogram to {hist_path}")
    
    plt.show()
else:
    print("No correlation files found for quality assessment.")

## Summary Statistics

In [ ]:
import pandas as pd

# Parse date pairs
date_pairs = parse_date_pairs_from_hyp3_filenames(unwrapped_files)

# Calculate temporal baselines
if date_pairs:
    temporal_baselines = [(sec - ref).days for ref, sec in date_pairs]
    
    stats_df = pd.DataFrame({
        "Reference Date": [ref.strftime('%Y-%m-%d') for ref, sec in date_pairs],
        "Secondary Date": [sec.strftime('%Y-%m-%d') for ref, sec in date_pairs],
        "Temporal Baseline (days)": temporal_baselines,
    })
    
    print(f"\nInterferogram Summary:")
    print(f"  Total pairs: {len(date_pairs)}")
    print(f"  Temporal baseline range: {min(temporal_baselines)} - {max(temporal_baselines)} days")
    print(f"  Mean temporal baseline: {np.mean(temporal_baselines):.1f} days")
    print(f"\nFirst 10 pairs:")
    display(stats_df.head(10))
else:
    print("Could not parse date pairs from filenames.")

## Next Steps

**For time series analysis:**
- Use MintPy to process stack into displacement time series
- See `MintPy_SNOTEL.ipynb` for comparison with ground truth

**For quality filtering:**
- Exclude pairs with mean correlation < 0.4
- Consider temporal baseline vs correlation tradeoff

**For publication figures:**
- Adjust `UNWRAPPED_VLIM_RAD` to highlight your displacement range
- Increase `SAVE_DPI` to 300 for print quality
- Modify colormaps (`cmap` parameter) for specific journals